# Análise exploratória do dataset NLP

Este notebook audita o dataset sintético multirrótulo antes do treinamento.

Verificações realizadas:

- estrutura e campos obrigatórios;
- quantidade de exemplos por split;
- distribuição dos rótulos;
- coocorrência entre rótulos;
- conflitos com `nao_violento`;
- duplicatas e vazamento entre splits;
- comprimento dos textos;
- registros linguísticos, regiões, palavrões e ruído de ASR;
- inspeção manual de exemplos;
- geração de relatório final.

In [ ]:
from pathlib import Path
import os

RAIZ_PROJETO = (
    Path.home()
    / "Documentos"
    / "projetos"
    / "violence-edge-ai"
)

os.chdir(RAIZ_PROJETO)

print("Diretório atual:", Path.cwd())

In [ ]:
!pip -q install pandas matplotlib tqdm

In [ ]:
from collections import Counter
import json
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import display
from tqdm.auto import tqdm

SEMENTE = 42
random.seed(SEMENTE)
np.random.seed(SEMENTE)

ROTULOS = [
    "pedido_socorro",
    "ameaca",
    "agressao",
    "referencia_arma",
    "nao_violento",
]

print("Ambiente preparado.")

## 1. Localizar os arquivos do dataset

O notebook procura automaticamente os arquivos em:

```text
datasets/nlp/splits/
```

In [ ]:
def encontrar_raiz_projeto() -> Path | None:
    candidatos = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/violence-edge-ai"),
    ]

    for candidato in candidatos:
        pasta_splits = candidato / "datasets" / "nlp" / "splits"

        if pasta_splits.exists():
            return candidato.resolve()

    return None


RAIZ_PROJETO = encontrar_raiz_projeto()

if RAIZ_PROJETO is not None:
    PASTA_SPLITS = RAIZ_PROJETO / "datasets" / "nlp" / "splits"
    PASTA_SAIDA = RAIZ_PROJETO / "outputs" / "metrics" / "nlp"
else:
    PASTA_SPLITS = Path("/content/dataset_nlp")
    PASTA_SAIDA = Path("/content/outputs_nlp")

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", RAIZ_PROJETO)
print("Pasta dos splits:", PASTA_SPLITS)
print("Pasta de saída:", PASTA_SAIDA)

## 2. Carregar os três splits

In [ ]:
def carregar_jsonl(caminho: Path, split: str) -> pd.DataFrame:
    registros = []

    with caminho.open("r", encoding="utf-8") as arquivo:
        linhas = arquivo.readlines()

    for numero_linha, linha in enumerate(
        tqdm(linhas, desc=f"Carregando {split}", unit="linha"),
        start=1,
    ):
        linha = linha.strip()

        if not linha:
            continue

        try:
            registro = json.loads(linha)
        except json.JSONDecodeError as erro:
            raise ValueError(
                f"JSON inválido em {caminho}, linha {numero_linha}: {erro}"
            ) from erro

        registro["split"] = split
        registros.append(registro)

    return pd.DataFrame(registros)


arquivos_splits = {
    "treino": PASTA_SPLITS / "treino.jsonl",
    "validacao": PASTA_SPLITS / "validacao.jsonl",
    "teste": PASTA_SPLITS / "teste.jsonl",
}

for nome, caminho in arquivos_splits.items():
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

tabelas = {
    nome: carregar_jsonl(caminho, nome)
    for nome, caminho in arquivos_splits.items()
}

dados = pd.concat(
    tabelas.values(),
    ignore_index=True,
)

print("Total de registros:", len(dados))
display(dados.head())

## 3. Validar estrutura e campos obrigatórios

In [ ]:
CAMPOS_OBRIGATORIOS = {
    "id",
    "texto",
    "rotulos",
    "perfil",
    "registro",
    "regiao",
    "possui_palavrao",
    "ruido_asr",
    "grupo_id",
    "split",
}


def campos_faltantes(registro: pd.Series) -> list[str]:
    return [
        campo
        for campo in CAMPOS_OBRIGATORIOS
        if campo not in registro.index
        or registro[campo] is None
    ]


linhas_com_campos_faltantes = []

for indice, registro in dados.iterrows():
    faltantes = campos_faltantes(registro)

    if faltantes:
        linhas_com_campos_faltantes.append(
            {
                "indice": indice,
                "id": registro.get("id"),
                "campos_faltantes": faltantes,
            }
        )

tabela_campos_faltantes = pd.DataFrame(linhas_com_campos_faltantes)

print("Registros com campos faltantes:", len(tabela_campos_faltantes))

if not tabela_campos_faltantes.empty:
    display(tabela_campos_faltantes.head(20))

## 4. Expandir os rótulos para colunas

In [ ]:
def extrair_rotulo(rotulos: dict, nome: str) -> int:
    if not isinstance(rotulos, dict):
        return -1

    valor = rotulos.get(nome, -1)

    try:
        return int(valor)
    except (TypeError, ValueError):
        return -1


for rotulo in ROTULOS:
    dados[rotulo] = dados["rotulos"].apply(
        lambda valor, nome=rotulo: extrair_rotulo(valor, nome)
    )

display(
    dados[
        ["id", "texto", "split", *ROTULOS]
    ].head()
)

## 5. Quantidade por split

In [ ]:
quantidade_por_split = (
    dados["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="quantidade")
)

quantidade_por_split["proporcao"] = (
    quantidade_por_split["quantidade"]
    / len(dados)
)

display(quantidade_por_split)

plt.figure(figsize=(8, 5))
plt.bar(
    quantidade_por_split["split"],
    quantidade_por_split["quantidade"],
)
plt.xlabel("Split")
plt.ylabel("Quantidade")
plt.title("Quantidade de exemplos por split")
plt.grid(axis="y")
plt.show()

## 6. Distribuição dos rótulos

In [ ]:
distribuicao_rotulos = (
    dados.groupby("split")[ROTULOS]
    .sum()
    .astype(int)
)

display(distribuicao_rotulos)

distribuicao_total = (
    dados[ROTULOS]
    .sum()
    .sort_values(ascending=False)
    .astype(int)
)

display(
    distribuicao_total
    .rename("quantidade")
    .to_frame()
)

plt.figure(figsize=(10, 5))
plt.bar(
    distribuicao_total.index,
    distribuicao_total.values,
)
plt.xlabel("Rótulo")
plt.ylabel("Quantidade de exemplos positivos")
plt.title("Distribuição total dos rótulos")
plt.xticks(rotation=25)
plt.grid(axis="y")
plt.show()

## 7. Coocorrência entre rótulos

In [ ]:
matriz_coocorrencia = pd.DataFrame(
    0,
    index=ROTULOS,
    columns=ROTULOS,
    dtype=int,
)

for primeiro in ROTULOS:
    for segundo in ROTULOS:
        matriz_coocorrencia.loc[primeiro, segundo] = int(
            (
                (dados[primeiro] == 1)
                & (dados[segundo] == 1)
            ).sum()
        )

display(matriz_coocorrencia)

plt.figure(figsize=(8, 7))
plt.imshow(matriz_coocorrencia.values)
plt.xticks(
    range(len(ROTULOS)),
    ROTULOS,
    rotation=45,
    ha="right",
)
plt.yticks(
    range(len(ROTULOS)),
    ROTULOS,
)
plt.title("Matriz de coocorrência dos rótulos")
plt.colorbar(label="Quantidade")

for linha in range(len(ROTULOS)):
    for coluna in range(len(ROTULOS)):
        plt.text(
            coluna,
            linha,
            matriz_coocorrencia.iloc[linha, coluna],
            ha="center",
            va="center",
        )

plt.tight_layout()
plt.show()

## 8. Verificar conflitos de rótulos

In [ ]:
rotulos_risco = [
    "pedido_socorro",
    "ameaca",
    "agressao",
    "referencia_arma",
]

dados["quantidade_rotulos_risco"] = (
    dados[rotulos_risco]
    .sum(axis=1)
)

conflitos_nao_violento = dados[
    (dados["nao_violento"] == 1)
    & (dados["quantidade_rotulos_risco"] > 0)
]

sem_rotulo_valido = dados[
    (dados["nao_violento"] == 0)
    & (dados["quantidade_rotulos_risco"] == 0)
]

valores_invalidos = dados[
    ~dados[ROTULOS]
    .isin([0, 1])
    .all(axis=1)
]

print(
    "Não violento junto com rótulo de risco:",
    len(conflitos_nao_violento),
)
print(
    "Sem nenhum rótulo válido:",
    len(sem_rotulo_valido),
)
print(
    "Rótulos fora de 0 ou 1:",
    len(valores_invalidos),
)

if not conflitos_nao_violento.empty:
    display(
        conflitos_nao_violento[
            ["id", "texto", *ROTULOS]
        ].head(20)
    )

## 9. Duplicatas e vazamento entre splits

São verificadas sobreposições por:

- `id`;
- texto exato normalizado;
- `grupo_id`.

O `grupo_id` é a verificação mais importante para frases sintéticas.

In [ ]:
def normalizar_texto(texto: str) -> str:
    texto = str(texto).lower().strip()
    texto = re.sub(r"\s+", " ", texto)
    return texto


dados["texto_normalizado"] = dados["texto"].apply(
    normalizar_texto
)

duplicatas_id = dados[
    dados.duplicated("id", keep=False)
].sort_values("id")

duplicatas_texto = dados[
    dados.duplicated("texto_normalizado", keep=False)
].sort_values("texto_normalizado")

print("IDs duplicados:", len(duplicatas_id))
print("Textos duplicados:", len(duplicatas_texto))

if not duplicatas_texto.empty:
    display(
        duplicatas_texto[
            ["id", "texto", "split", "grupo_id"]
        ].head(20)
    )

In [ ]:
def sobreposicao_entre_splits(
    coluna: str,
    split_a: str,
    split_b: str,
) -> set:
    valores_a = set(
        dados.loc[
            dados["split"] == split_a,
            coluna,
        ].dropna()
    )

    valores_b = set(
        dados.loc[
            dados["split"] == split_b,
            coluna,
        ].dropna()
    )

    return valores_a & valores_b


pares_splits = [
    ("treino", "validacao"),
    ("treino", "teste"),
    ("validacao", "teste"),
]

resultados_vazamento = []

for coluna in (
    "id",
    "texto_normalizado",
    "grupo_id",
):
    for split_a, split_b in pares_splits:
        sobreposicao = sobreposicao_entre_splits(
            coluna,
            split_a,
            split_b,
        )

        resultados_vazamento.append(
            {
                "campo": coluna,
                "split_a": split_a,
                "split_b": split_b,
                "quantidade_sobreposta": len(sobreposicao),
            }
        )

tabela_vazamento = pd.DataFrame(
    resultados_vazamento
)

display(tabela_vazamento)

## 10. Comprimento dos textos

In [ ]:
dados["quantidade_palavras"] = (
    dados["texto"]
    .astype(str)
    .str.split()
    .str.len()
)

dados["quantidade_caracteres"] = (
    dados["texto"]
    .astype(str)
    .str.len()
)

resumo_comprimento = dados[
    [
        "quantidade_palavras",
        "quantidade_caracteres",
    ]
].describe()

display(resumo_comprimento)

plt.figure(figsize=(10, 5))
plt.hist(
    dados["quantidade_palavras"],
    bins=30,
)
plt.xlabel("Quantidade de palavras")
plt.ylabel("Quantidade de textos")
plt.title("Distribuição do comprimento dos textos")
plt.grid(axis="y")
plt.show()

textos_muito_curtos = dados[
    dados["quantidade_palavras"] <= 2
]

textos_muito_longos = dados[
    dados["quantidade_palavras"] >= 30
]

print("Textos com até 2 palavras:", len(textos_muito_curtos))
print("Textos com 30 palavras ou mais:", len(textos_muito_longos))

display(
    textos_muito_curtos[
        ["texto", "perfil", "split"]
    ].head(20)
)

## 11. Distribuição linguística

In [ ]:
def tabela_frequencia(coluna: str) -> pd.DataFrame:
    return (
        dados[coluna]
        .value_counts(dropna=False)
        .rename_axis(coluna)
        .reset_index(name="quantidade")
    )


for coluna in (
    "perfil",
    "registro",
    "regiao",
    "possui_palavrao",
    "ruido_asr",
):
    print(f"\nDistribuição: {coluna}")
    display(tabela_frequencia(coluna))

In [ ]:
distribuicao_registro = tabela_frequencia(
    "registro"
)

plt.figure(figsize=(10, 5))
plt.bar(
    distribuicao_registro["registro"].astype(str),
    distribuicao_registro["quantidade"],
)
plt.xlabel("Registro linguístico")
plt.ylabel("Quantidade")
plt.title("Distribuição dos registros linguísticos")
plt.xticks(rotation=25)
plt.grid(axis="y")
plt.show()

distribuicao_regiao = tabela_frequencia(
    "regiao"
)

plt.figure(figsize=(10, 5))
plt.bar(
    distribuicao_regiao["regiao"].astype(str),
    distribuicao_regiao["quantidade"],
)
plt.xlabel("Região ou marcador")
plt.ylabel("Quantidade")
plt.title("Distribuição regional")
plt.xticks(rotation=25)
plt.grid(axis="y")
plt.show()

## 12. Relação entre linguagem e rótulos

Esta verificação ajuda a descobrir se uma característica artificial aparece concentrada em uma classe.

In [ ]:
def distribuicao_por_caracteristica(
    coluna: str,
) -> pd.DataFrame:
    linhas = []

    for valor, grupo in dados.groupby(
        coluna,
        dropna=False,
    ):
        linha = {
            coluna: valor,
            "quantidade": len(grupo),
        }

        for rotulo in ROTULOS:
            linha[rotulo] = int(
                grupo[rotulo].sum()
            )

        linhas.append(linha)

    return pd.DataFrame(linhas)


display(
    distribuicao_por_caracteristica(
        "registro"
    )
)

display(
    distribuicao_por_caracteristica(
        "regiao"
    )
)

display(
    distribuicao_por_caracteristica(
        "ruido_asr"
    )
)

## 13. Inspeção manual de exemplos

Exibe exemplos aleatórios por perfil.

In [ ]:
def mostrar_exemplos_por_perfil(
    quantidade_por_perfil: int = 10,
) -> None:
    for perfil in sorted(
        dados["perfil"].dropna().unique()
    ):
        exemplos = dados[
            dados["perfil"] == perfil
        ].sample(
            n=min(
                quantidade_por_perfil,
                len(
                    dados[
                        dados["perfil"] == perfil
                    ]
                ),
            ),
            random_state=SEMENTE,
        )

        print("\n" + "=" * 80)
        print("PERFIL:", perfil)
        print("=" * 80)

        for _, exemplo in exemplos.iterrows():
            ativos = [
                rotulo
                for rotulo in ROTULOS
                if exemplo[rotulo] == 1
            ]

            print(
                f"[{exemplo['split']}] "
                f"{exemplo['texto']}"
            )
            print(
                "Rótulos:",
                ", ".join(ativos),
            )
            print(
                "Registro:",
                exemplo["registro"],
                "| Região:",
                exemplo["regiao"],
                "| ASR:",
                exemplo["ruido_asr"],
            )
            print("-" * 80)


mostrar_exemplos_por_perfil(
    quantidade_por_perfil=5
)

## 14. Salvar relatório da auditoria

In [ ]:
relatorio = {
    "quantidade_total": int(len(dados)),
    "quantidade_por_split": {
        linha["split"]: int(linha["quantidade"])
        for _, linha in quantidade_por_split.iterrows()
    },
    "distribuicao_rotulos": {
        rotulo: int(distribuicao_total[rotulo])
        for rotulo in ROTULOS
    },
    "problemas": {
        "campos_faltantes": int(
            len(tabela_campos_faltantes)
        ),
        "conflitos_nao_violento": int(
            len(conflitos_nao_violento)
        ),
        "sem_rotulo_valido": int(
            len(sem_rotulo_valido)
        ),
        "valores_invalidos": int(
            len(valores_invalidos)
        ),
        "ids_duplicados": int(
            len(duplicatas_id)
        ),
        "textos_duplicados": int(
            len(duplicatas_texto)
        ),
    },
    "vazamento": tabela_vazamento.to_dict(
        orient="records"
    ),
    "comprimento_textos": {
        "media_palavras": float(
            dados["quantidade_palavras"].mean()
        ),
        "mediana_palavras": float(
            dados["quantidade_palavras"].median()
        ),
        "minimo_palavras": int(
            dados["quantidade_palavras"].min()
        ),
        "maximo_palavras": int(
            dados["quantidade_palavras"].max()
        ),
    },
    "distribuicao_registro": {
        str(chave): int(valor)
        for chave, valor in dados[
            "registro"
        ].value_counts(dropna=False).items()
    },
    "distribuicao_regiao": {
        str(chave): int(valor)
        for chave, valor in dados[
            "regiao"
        ].value_counts(dropna=False).items()
    },
}

caminho_relatorio = (
    PASTA_SAIDA
    / "relatorio_analise_dataset_nlp.json"
)

caminho_relatorio.write_text(
    json.dumps(
        relatorio,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

distribuicao_rotulos.to_csv(
    PASTA_SAIDA
    / "distribuicao_rotulos_por_split.csv"
)

tabela_vazamento.to_csv(
    PASTA_SAIDA
    / "verificacao_vazamento.csv",
    index=False,
)

matriz_coocorrencia.to_csv(
    PASTA_SAIDA
    / "matriz_coocorrencia.csv"
)

print("Relatório salvo em:", caminho_relatorio)